# Reproducing Chaunsali et al. (2025) from dandiset 001702

Paper: Chaunsali L, Li J, Fleischel E, et al. "Degradation of perineuronal nets in hippocampal CA2
explains the loss of social cognition memory in Alzheimer's disease". Alzheimer's & Dementia 21,
e70813 (2025), doi:10.1002/alz.70813.

Dataset: DANDI:001702, "Hippocampal CA2 Perineuronal Net Degradation and Social Memory Deficits in a
5XFAD Alzheimer's Disease Mouse Model", version **0.260216.2323** (doi:10.48324/dandi.001702/0.260216.2323).
The paper cites no version of the dandiset. This is its only published version, and the draft
(modified 2026-04-10) has the same 656 assets and size; the most recent published version is
pinned, as the protocol asks when the paper cites none.

Claims: `claims/001702/10.1002-alz.70813.yaml`, written from the version of record (Europe PMC,
PMC12541281): main text, Methods, the legends of Figures 1 to 5. The Supporting Information was
not in the text read.

Outside data (claims marked `data: elsewhere`): the bulk RNA-seq counts are not in the dandiset.
They are in GEO series GSE317499, deposited by the paper's authors, which the dandiset's metadata
links as a related resource; the file `GSE317499_counts.csv.gz` is read and checked against its
SHA-256. GO annotations are from the Gene Ontology release 2024-06-17 (doi:10.5281/zenodo.12173881).

| Claims | Status |
|---|---|
| Figure 1, Figures S2 to S6 (IHC: PNN area and intensity, PV and NeuN density, PCP4, other regions, J20 and 3XTg, aggrecan, plaques) | blocked: the NWB files hold no image data |
| Figure 3B, 3G, Figures S1, S9, Figure 5I, 5 M PNNs (IHC) | blocked: the NWB files hold no image data |
| Figures 2, 3, 5 and S8 social memory, Figures 2I to 2K and S7 object recognition | blocked: videos without subject, group or trial labels, and no scored times |
| fig4a-7m-social-memory-deficit | blocked: no behavior of the RNA-seq cohort |
| Figure 4B to 4I (RNA-seq, 9 claims) | tested on GEO GSE317499 (`data: elsewhere`) |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import warnings
warnings.filterwarnings("ignore")
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ca2pnn as c

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
print("dandiset", c.DANDISET, "version", c.VERSION)

## Data check

What the dandiset holds: every asset of the pinned version, the content of every NWB file (all
481 are opened and every dataset outside `/general` and `/specifications` is listed), the videos,
and the GEO counts used for Figure 4.

In [ ]:
meta = c.version_metadata()
print("name:", meta["name"])
print("description:", meta["description"])
print("related resources:", [(r.get("identifier"), r.get("relation"), r.get("resourceType")) for r in meta.get("relatedResource", [])])
print("assetsSummary: subjects", meta["assetsSummary"].get("numberOfSubjects"), "; files", meta["assetsSummary"].get("numberOfFiles"),
      "; variableMeasured", meta["assetsSummary"].get("variableMeasured"), "; approach", meta["assetsSummary"].get("approach"))
assets = c.list_assets()
print(len(assets), "assets;", assets.ext.value_counts().to_dict(), "; total", round(assets["size"].sum() / 1e9, 2), "GB")
nwb = c.scan_all_nwb(assets)
print("NWB file sizes (bytes):", sorted(nwb["size"].unique()))
print("datasets outside /general and /specifications, by distinct set:")
print(nwb.datasets.value_counts().to_string())
print("data groups:", nwb.groups.value_counts().to_dict())
print("stimulus members:", nwb.stimulus_members.value_counts().to_dict())
print("NWB files holding any data object (acquisition, processing, analysis, stimulus):", int((nwb.n_data_objects > 0).sum()), "of", len(nwb))

In [ ]:
# What the NWB metadata says each file is: figure and panel from the keywords, genotype, age, sex.
nwb["figure"] = nwb.folder.str.extract(r"-(Fig\d|SF\d)")[0]
nwb["panel"] = nwb.identifier.str.extract(r"^(?:Sup )?Fig ?(\d+[A-Z]?)")[0]
summary = (nwb.groupby(["figure", "folder"])
           .agg(files=("path", "size"), genotype=("genotype", lambda s: ",".join(sorted(set(s)))),
                age=("age", lambda s: ",".join(sorted(set(s)))), sex=("sex", lambda s: ",".join(sorted(set(s)))),
                panels=("panel", lambda s: ",".join(sorted(set(s.dropna())))),
                subject_ids=("subject_id", "nunique")))
print(summary.to_string())
print()
print("distinct subject_id values:", nwb.subject_id.nunique(), "for", len(nwb), "files; files per subject_id: max",
      nwb.subject_id.value_counts().max())
print("sex values over all files:", nwb.sex.value_counts().to_dict())
print("session years:", nwb.path.str.extract(r"ses-(\d{4})")[0].value_counts(dropna=False).to_dict())
print("example identifiers (the original image file names):")
print(nwb.identifier.head(5).to_string())

Each NWB file is 185,480 bytes and holds only file-level metadata: an identifier that is the
original image's file name, a session description, keywords with the figure panel and stains,
and a subject (genotype, age, sex). No file holds an image, an ROI, a table or a measurement, so
none of the IHC quantities (PNN area, WFA intensity, PV or NeuN density, PCP4, aggrecan, 2B6,
amyloid) can be computed. Several folders also name groups and not animals: the `subject_id` is
the group label (for example `5xFAD+_6M_Fig1` for 31 files).

In [ ]:
# The behaviour videos: MP4 files outside NWB, named by figure panel and trial number only.
vid = assets[assets.ext == "mp4"].copy()
vid["group"] = vid.path.str.extract(r"/(.*)_Trial\d+\.mp4$")[0]
vid["trial"] = vid.path.str.extract(r"_Trial(\d+)\.mp4$")[0].astype(int)
# Mice per group, from the claims file's description of the paper (legends of Figures 2, 3, 5).
paper_n = {"Fig 2B,C,D": 16, "Fig 2E,F,G": 19, "Fig 2I": 28, "Fig 2J": 28, "Fig 2K": 39, "Fig 3C,D,E": 20,
           "Fig 3H,I,J": 18, "Fig 3K,L,M": 18, "Fig 5B,C,D": 18, "Fig 5E,F,G": 26}
probe = {g: c.probe_video(vid[vid.group == g].s3.iloc[0]) for g in sorted(vid.group.unique())}
tab = vid.groupby("group").agg(videos=("path", "size"), unique_files=("sha256", "nunique"))
tab["mice_in_paper"] = pd.Series(paper_n)
for k in ("width", "height", "fps", "duration_s"):
    tab[k + "_first_video"] = pd.Series({g: p[k] for g, p in probe.items()})
print(tab.to_string())
dups = vid.groupby("sha256").path.apply(list)
dups = dups[dups.str.len() > 1]
print()
print(len(vid), "videos,", vid.sha256.nunique(), "distinct files;", len(dups), "sets of byte-identical videos:")
for d in dups:
    print("  ", d)

In [ ]:
# First frame of the first Figure 2B to 2D video: four arenas side by side, one card per arena.
fr = c.probe_video(vid[vid.group == "Fig 2B,C,D"].s3.iloc[0], frame=True)["frame"]
print("frame shape", fr.shape, "pixel range", fr.min(), fr.max())
plt.figure(figsize=(10, 4))
plt.imshow(fr)
plt.axis("off")
plt.title("Fig 2B,C,D_Trial1.mp4, first frame")
plt.show()

The 175 videos have no NWB wrapper and no metadata beyond their names: no subject, genotype,
treatment, DPI, trial type (Novel 1, Familiar, Novel 2; familiarization or test) or start time.
The frames show several arenas at once, each with a handwritten card, so videos do not map one
to one to mice or trials, and no table in the dandiset links them to the paper's groups. The
interaction and exploration times the paper scored (by hand for social interaction, Methods
2.6.1; with Ethovision XT for objects, Methods 2.6.2) are not deposited. The object recognition
videos of Figures 2I, 2J, 2K and S7 are the same seven files deposited four times, and within
each set Trial 6 and Trial 7 are the same file.

There are no time series in the dataset, so there is no clock to check against a trial table.

In [ ]:
counts, samples = c.load_counts()
geo = c.geo_sample_records()
print("GEO counts:", counts.shape[0], "genes x", counts.shape[1], "samples; integer:", bool((counts.dtypes == "int64").all()),
      "; duplicated gene names:", int(counts.index.duplicated().sum()), "; sha256 checked:", c.GEO_COUNTS_SHA256[:16])
print(samples.group.value_counts().sort_index().to_dict())
lib = counts.sum().rename("library_size")
print("library sizes: min", int(lib.min()), "median", int(lib.median()), "max", int(lib.max()))
# Check the column names against GEO's own sample records, matched by the sample number that leads
# each run name.
m = samples.reset_index().merge(geo, on="number", how="left")
m["geo_age"] = m.title.str.extract(r"^(\d) Month")[0].radd("M")
m["geo_geno"] = np.where(m.genotype.str.contains("Positive"), "pos", "neg")
m["run_library"] = m.run_name.str.split("_").str[3]
print("columns matched to a GEO sample:", int(m.gsm.notna().sum()), "of", len(m),
      "; age agrees:", int((m.geo_age == m.age).sum()), "; genotype agrees:", int((m.geo_geno == m.geno).sum()))
print("library name differs from the number in the run name:")
print(m.loc[m.library != m.run_library, ["index", "gsm", "title", "library", "run_name"]].to_string())
print("batch values:", geo.batch.value_counts().to_dict())

The GEO matrix has 16 samples, 4 per age and genotype; the column names agree with GEO's sample
records on age and genotype. The paper does not give the number of mice per group (Table S4 was
not read). The sample size of 4 per group is from GEO.

## Figure 4: bulk RNA-seq of CA2 (GEO GSE317499)

DESeq2 1.44 in R is replaced by pydeseq2, its Python port, with DESeq2's defaults. R's
results() tunes independent filtering at alpha 0.1, while pydeseq2's DeseqStats defaults to 0.05,
so alpha 0.1 is set explicitly; filtering at 0.05 is shown as a variant for the DEG counts. The
claims file names the readings: (a) one model per contrast on its own samples, or (b) one model on
all 16 samples with a four-level group factor; each with (c) unshrunken log2 fold changes or (d)
apeglm-type shrinkage. All four are run. The Methods do not say which model was used (Methods 2.5
names only DESeq2 1.44.0). Following the claims file's READINGS rule, each claim's outcome is the
one all readings give, and the claim is blocked as methods_underspecified when they give different
outcomes. Each claim below prints the outcome under every reading.


In [ ]:
import time
c3 = list(samples.index[samples.age == "M3"])
c7 = list(samples.index[samples.age == "M7"])
cpos = list(samples.index[samples.geno == "pos"])
call = list(samples.index)
specs = {
    ("a", "3M"): (c3, "geno", "pos", "neg"), ("a", "7M"): (c7, "geno", "pos", "neg"), ("a", "age"): (cpos, "age", "M7", "M3"),
    ("b", "3M"): (call, "group", "M3_pos", "M3_neg"), ("b", "7M"): (call, "group", "M7_pos", "M7_neg"),
    ("b", "age"): (call, "group", "M7_pos", "M3_pos"),
}
fits = {}
for key, args in specs.items():
    t = time.time()
    fits[key] = c.deseq(counts, samples, *args)
    r = fits[key]
    print(key, "samples", len(args[0]), "; genes with a p-value", int(r.pvalue.notna().sum()),
          "; with an adjusted p", int(r.padj.notna().sum()), f"; {time.time() - t:.0f} s")
CONTRAST = {"3M": "3 M 5XFAD vs control", "7M": "7 M 5XFAD vs control", "age": "7 M 5XFAD vs 3 M 5XFAD"}
MODEL = {"a": "separate", "b": "joint"}
LFC = {"c": "lfc", "d": "lfc_shrunk"}

### fig4b-3m-degs

> In CA2 of 3 M 5XFAD mice compared with controls, 98 genes are upregulated and 133 downregulated.
(expected 98 ± 9.8 and 133 ± 13.3; DEG: adjusted p < 0.05 and |log2 FC| > 1, Methods 2.5)

In [ ]:
def deg_table(contrast, padj_col="padj"):
    rows = []
    for m in "ab":
        for l in "cd":
            up, dn = c.degs(fits[(m, contrast)], LFC[l], padj_col)
            rows.append(dict(reading=f"({m}) {MODEL[m]}, ({l}) {'unshrunken' if l == 'c' else 'shrunken'}", up=len(up), down=len(dn)))
    return pd.DataFrame(rows)

def outcome_4b(t):
    full = t["up_within_98±9.8"] & t["down_within_133±13.3"]
    part = t.more_down_than_up & (t["up_within_98±9.8"] | t["down_within_133±13.3"])
    return np.where(full, "reproduced", np.where(part, "partially_reproduced", "not_reproduced"))

for padj_col, title in (("padj", "independent filtering at alpha 0.1 (R results() default, used)"),
                        ("padj_filter005", "variant, independent filtering at alpha 0.05 (pydeseq2 default)")):
    t3 = deg_table("3M", padj_col)
    t3["up_within_98±9.8"] = (t3.up - 98).abs() <= 9.8
    t3["down_within_133±13.3"] = (t3.down - 133).abs() <= 13.3
    t3["more_down_than_up"] = t3.down > t3.up
    t3["outcome"] = outcome_4b(t3)
    print(title); print(t3.to_string(index=False)); print()
r = fits[("b", "3M")]
print("variant, adjusted p < 0.05 at any fold change, joint model:", int((r.padj < 0.05).sum()))
for thr in (0.5, 1.0, 1.5):
    sig = r.padj < 0.05
    print(f"  joint, unshrunken, |log2FC| > {thr}: up {int((sig & (r.lfc > thr)).sum())}, down {int((sig & (r.lfc < -thr)).sum())}")

**Result: blocked, methods_underspecified: the readings give different outcomes.** With one model
per contrast and unshrunken fold changes the counts are 95 up and 136 down, both within tolerance
(reproduced). With one model on all 16 samples, unshrunken, they are 51 and 141: the down count is
within tolerance and more genes go down than up (partially reproduced). With shrunken fold changes
they are 88 and 45 (separate) and 39 and 37 (joint), more up than down or nearly equal (not
reproduced). Filtering at pydeseq2's alpha 0.05 gives 98 and 137 for the separate unshrunken
reading and the same outcomes per reading.

### fig4c-7m-degs

> In CA2 of 7 M 5XFAD mice compared with controls, 314 genes are upregulated and 6 downregulated.
(expected 314 ± 31.4 and 6 ± 0.6)

In [ ]:
for padj_col, title in (("padj", "independent filtering at alpha 0.1 (R results() default, used)"),
                        ("padj_filter005", "variant, independent filtering at alpha 0.05 (pydeseq2 default)")):
    t7 = deg_table("7M", padj_col)
    t7["up_within_314±31.4"] = (t7.up - 314).abs() <= 31.4
    t7["down_within_6±0.6"] = (t7.down - 6).abs() <= 0.6
    t7["down_below_20"] = t7.down < 20
    t7["outcome"] = np.where(t7["up_within_314±31.4"] & t7["down_within_6±0.6"], "reproduced",
                             np.where(t7["up_within_314±31.4"] & t7.down_below_20, "partially_reproduced", "not_reproduced"))
    print(title); print(t7.to_string(index=False)); print()

**Result: partially reproduced.** Under all four readings the up count is within 314 ± 31.4
(284 to 320) and the down count is below 20 (0 to 7), but no down count is within 6 ± 0.6. One
model per contrast gives 317 and 7 unshrunken and 284 and 3 shrunken; one model on all samples
gives 320 and 2 unshrunken and 284 and 0 shrunken. Filtering at alpha 0.05 gives the same outcome
under every reading.

### GO enrichment (Figures 4D to 4G)

Over-representation of each contrast's DEGs (up and down together) in GO Biological Process,
one-sided hypergeometric test, Benjamini-Hochberg; backgrounds (a) genes tested by DESeq2 (a
p-value not NA) and (b) all BP-annotated mouse genes, as the criterion names. The paper names R
and GOplot but no enrichment tool (Methods 2.5), so the term-size limit is open too, and two
readings are run: terms of 10 to 500 genes (clusterProfiler enrichGO's defaults) and no size
limit. With the two DESeq2 readings of each kind, each GO claim has 16 readings. GOplot's z-score
is (up - down) / sqrt(count).


In [ ]:
go_names, gene_terms = c.load_go()
print("GO release", c.GO_RELEASE, ":", len(go_names), "BP terms;", len(gene_terms), "mouse genes with BP annotations")
measured = set(counts.index)
print("count-matrix genes with BP annotations:", len(measured & set(gene_terms)), "of", len(measured))
BG = {"a": "tested", "b": "all annotated"}
go = {}
for m in "ab":
    for l in "cd":
        for k in ("3M", "7M", "age"):
            r = fits[(m, k)]
            up, dn = c.degs(r, LFC[l])
            for bg in "ab":
                univ = set(r.index[r.pvalue.notna()]) if bg == "a" else set(gene_terms)
                for size in ("10-500", "none"):
                    go[(m, l, k, bg, size)] = c.enrich(up, dn, univ, gene_terms, go_names,
                                                        size=(c.GO_MIN, c.GO_MAX) if size == "10-500" else None)
print(len(go), "enrichment runs")

def top10_count(key, pattern, sign):
    top = go[key].head(10)
    ok = top.name.str.contains(pattern) & (np.sign(top.zscore) == sign)
    return int(ok.sum()), top

def label(m, l, bg, size):
    return f"({m}) {MODEL[m]}, ({l}) {'unshrunken' if l == 'c' else 'shrunken'}, bg ({bg}) {BG[bg]}, size {size}"

### fig4d-3m-go-memory-terms-down

> Among the top 10 GO terms at 3 M, many pathways of learning, memory and cognition are
> downregulated in 5XFAD mice.

In [ ]:

def grade(n):
    return "reproduced" if n >= 3 else ("partially_reproduced" if n >= 1 else "not_reproduced")

rows = []
for (m, l, k, bg, size), e in go.items():
    if k != "3M":
        continue
    n, top = top10_count((m, l, k, bg, size), c.MEMORY, -1)
    rows.append(dict(reading=label(m, l, bg, size), terms_tested=len(e), significant=int((e.padj < 0.05).sum()),
                     memory_terms_down_in_top10=n, outcome=grade(n)))
t4d = pd.DataFrame(rows)
print(t4d.to_string(index=False))
print("outcomes over the 16 readings:", t4d.outcome.value_counts().to_dict())
print()
print("top 10 under every reading (memory = name matches learning, memory or cognition):")
for (m, l, k, bg, size), e in go.items():
    if k == "3M":
        top = e.head(10)[["name", "size", "count", "up", "down", "padj", "zscore"]].assign(memory=lambda d: d.name.str.contains(c.MEMORY))
        print(label(m, l, bg, size)); print(top.to_string()); print()

**Result: blocked, methods_underspecified: the readings give different outcomes.** Over the 16
readings (two DESeq2 models, two fold changes, two backgrounds, two term-size limits) the top 10
holds three learning, memory or cognition terms with negative z-scores in 2 (reproduced), one or two
in 5 (partially reproduced) and none in 9 (not reproduced). Three are found only with one model per
contrast, unshrunken fold changes and terms of 10 to 500 genes, under either background ("learning
or memory", "learning", "cognition"). Without a size limit, larger terms such as synaptic signalling
fill the top 10 and no reading has more than one.

### fig4e-7m-go-immune-terms-up

> Among the top 10 GO terms at 7 M, many immune response pathways are upregulated in 5XFAD mice.

In [ ]:
rows = []
for (m, l, k, bg, size), e in go.items():
    if k != "7M":
        continue
    n, top = top10_count((m, l, k, bg, size), c.IMMUNE, 1)
    rows.append(dict(reading=label(m, l, bg, size), terms_tested=len(e), significant=int((e.padj < 0.05).sum()),
                     immune_terms_up_in_top10=n, min_z_top10=round(top.zscore.min(), 2), outcome=grade(n)))
t4e = pd.DataFrame(rows)
print(t4e.to_string(index=False))
print("outcomes over the 16 readings:", t4e.outcome.value_counts().to_dict())
print()
print("top 10 under every reading (immune = name matches the IMMUNE pattern):")
for (m, l, k, bg, size), e in go.items():
    if k == "7M":
        top = e.head(10)[["name", "size", "count", "up", "down", "padj", "zscore"]].assign(immune=lambda d: d.name.str.contains(c.IMMUNE))
        print(label(m, l, bg, size)); print(top.to_string()); print()

**Result: reproduced.** Under all 16 readings, five to eight of the top 10 terms name immune or
defense responses or immune activation with positive z-scores, and the smallest z-score in any top
10 is 4.47. The top-10 lists of every reading are printed above.

### fig4fg-ecm-go-terms

> ECM organization, assembly, disassembly and secretion terms change in 7 M 5XFAD against 3 M
> 5XFAD and against 7 M controls, but not in 3 M 5XFAD against 3 M controls.

In [ ]:
rows = []
for (m, l, k, bg, size), e in go.items():
    ecm = e[e.name.str.contains(c.ECM)]
    rows.append(dict(reading=label(m, l, bg, size), contrast=CONTRAST[k], ecm_terms_tested=len(ecm),
                     ecm_terms_padj_below_005=int((ecm.padj < 0.05).sum()),
                     smallest_ecm_padj=float(ecm.padj.min()) if len(ecm) else np.nan,
                     best_ecm_term=ecm.name.iloc[0] if len(ecm) else ""))
ecm_tab = pd.DataFrame(rows)
print(ecm_tab.to_string(index=False))
print()
print("smallest ECM-term adjusted p over all readings and size choices, by contrast:",
      ecm_tab.groupby("contrast").smallest_ecm_padj.min().round(4).to_dict())
print("readings with an ECM term at adjusted p < 0.05:")
print(ecm_tab[ecm_tab.ecm_terms_padj_below_005 > 0][["reading", "contrast", "smallest_ecm_padj"]].to_string(index=False))
print()
# Outcome per reading: both 7 M contrasts with an ECM term at adjusted p < 0.05 and none at 3 M.
hit = ecm_tab.assign(h=ecm_tab.ecm_terms_padj_below_005 > 0).pivot(index="reading", columns="contrast", values="h")
n7 = hit[CONTRAST["7M"]].astype(int) + hit[CONTRAST["age"]].astype(int)
hit["outcome"] = np.where(hit[CONTRAST["3M"]], "not_reproduced",
                          np.where(n7 == 2, "reproduced", np.where(n7 == 1, "partially_reproduced", "not_reproduced")))
print(hit.to_string())
print("outcomes over the 16 readings:", hit.outcome.value_counts().to_dict())
print()
print("any term containing 'extracellular matrix' with adjusted p < 0.05, any reading:",
      int(sum(((e.name.str.contains("extracellular matrix")) & (e.padj < 0.05)).sum() for e in go.values())))
for k in ("3M", "7M", "age"):
    e = go[("b", "c", k, "a", "10-500")]
    ecm = e[e.name.str.contains("extracellular")]
    print(CONTRAST[k], "reading (b) joint, (c) unshrunken, background (a), size 10-500, terms containing 'extracellular':")
    print(ecm[["name", "size", "count", "up", "down", "p", "padj"]].head(6).to_string())

**Result: blocked, methods_underspecified: the readings give different outcomes.** In 15 of the 16
readings no term naming extracellular matrix organization, assembly, disassembly or secretion
reaches adjusted p < 0.05 in either 7 M contrast (not reproduced). In one, one model per contrast
with shrunken fold changes, all annotated genes as background and no term-size limit,
"extracellular matrix organization" reaches 0.048 in 7 M against 3 M 5XFAD, which gives partially
reproduced. In 7 M 5XFAD against control the smallest adjusted p of an ECM term under any reading is
0.061. The 3 M contrast has none under any reading (smallest 0.145), as the claim says. The Figure 4F and 4G legend ranks the top 10 terms by adjusted p
without saying they were significant, so whether the criterion's adjusted p < 0.05 matches the
paper is asked of a reader (`questions/001702.yaml`); the claim is tested as the claims file has it.

### Gene-level directions (Figures 4H and 4I)

In [ ]:
def gene_signs(genes, contrast):
    rows = []
    for g in genes:
        row = dict(gene=g, in_matrix=g in counts.index)
        for m in "ab":
            r = fits[(m, contrast)]
            if g in r.index:
                row[f"lfc_{m}c"] = r.loc[g, "lfc"]
                row[f"lfc_{m}d"] = r.loc[g, "lfc_shrunk"]
                row[f"padj_{m}"] = r.loc[g, "padj"]
        rows.append(row)
    return pd.DataFrame(rows)

def count_dir(tab, sign):
    return {f"({m}) {MODEL[m]}, ({l})": int((np.sign(tab[f"lfc_{m}{l}"]) == sign).sum()) for m in "ab" for l in "cd"}

### fig4h-pnn-synthesis-genes

> PNN synthesis genes are broadly stable; at 3 M, Vcan, Cspg5, Tnr and Hapln1 are downregulated
> in 5XFAD, and at 7 M, Vcan and Bgn are increased.

In [ ]:
h3 = gene_signs(["Vcan", "Cspg5", "Tnr", "Hapln1"], "3M")
h7 = gene_signs(["Vcan", "Bgn"], "7M")
print("3 M (expected lower):"); print(h3.to_string(index=False))
print("7 M (expected higher):"); print(h7.to_string(index=False))
d3, d7 = count_dir(h3, -1), count_dir(h7, 1)
print("genes in the named direction, of 6:", {k: d3[k] + d7[k] for k in d3})
pnn = ["Acan", "Bcan", "Ncan", "Vcan", "Hapln1", "Hapln4", "Tnr", "Cspg5", "Bgn", "Has1", "Has2", "Has3"]
print("other PNN genes, DEGs under reading (b) joint, (c) unshrunken (adjusted p < 0.05 and |log2FC| > 1):",
      {k: sorted(set(pnn) & (lambda u, d: u | d)(*c.degs(fits[("b", k)]))) for k in ("3M", "7M")})

**Result: reproduced.** Vcan, Cspg5, Tnr and Hapln1 have negative log2 fold changes at 3 M and
Vcan and Bgn positive ones at 7 M under every reading, 6 of 6. The changes are small (unshrunken
-0.24 to -0.13 at 3 M, 0.16 to 0.49 at 7 M), none of the six is a DEG, and only Bgn at 7 M has an
adjusted p below 0.05 (0.009 separate, 0.023 joint).

### fig4i-3m-remodeling-genes-up

> At 3 M, 5XFAD CA2 shows increased expression of Mmp2, 3, 9, 14, 15 and 17, Timp1, 2 and 3, and
> Adam10, 11, 15 and 17.

In [ ]:
g3 = ["Mmp2", "Mmp3", "Mmp9", "Mmp14", "Mmp15", "Mmp17", "Timp1", "Timp2", "Timp3", "Adam10", "Adam11", "Adam15", "Adam17"]
i3 = gene_signs(g3, "3M")
print(i3.to_string(index=False))
print("positive, of 13:", count_dir(i3, 1))

**Result: reproduced.** All 13 genes have positive log2 fold changes at 3 M under every reading.
The criterion judges the sign only, and most changes are near zero: apart from Mmp3 (unshrunken
2.04 to 2.07 but shrunken 0.002, with no adjusted p, which DESeq2's independent filtering sets to
NA) the unshrunken values are 0.003 to 0.71, and only Mmp2 in the separate model has an adjusted p
below 0.05 (0.038). Adam10 is +0.003 to +0.005 and Timp2 +0.07 (adjusted p 0.92).

### fig4i-7m-remodeling-genes-up

> At 7 M, 5XFAD CA2 shows increased expression of Mmp2, 3 and 16, Timp1 and 3, and Adam15 and 17.

In [ ]:
g7 = ["Mmp2", "Mmp3", "Mmp16", "Timp1", "Timp3", "Adam15", "Adam17"]
i7 = gene_signs(g7, "7M")
print(i7.to_string(index=False))
print("positive, of 7:", count_dir(i7, 1))

**Result: reproduced.** All 7 genes have positive log2 fold changes at 7 M under every reading
(unshrunken 0.10 to 1.68). Only Timp1 has an adjusted p below 0.05 (0.012 separate, 0.0007 joint);
Mmp3 has none after independent filtering.

### fig4i-common-and-7m-vs-3m-up

> Mmp2, Mmp3 and Timp1 are upregulated at both 3 and 7 M, and 7 M 5XFAD mice show higher Mmp3,
> Timp1, 2 and 3, and Adam10 and 17 than 3 M 5XFAD mice.

In [ ]:
common = pd.concat([gene_signs(["Mmp2", "Mmp3", "Timp1"], "3M").assign(contrast="3M"),
                    gene_signs(["Mmp2", "Mmp3", "Timp1"], "7M").assign(contrast="7M"),
                    gene_signs(["Mmp3", "Timp1", "Timp2", "Timp3", "Adam10", "Adam17"], "age").assign(contrast="age")])
print(common.to_string(index=False))
print("positive, of 12:", count_dir(common, 1))

**Result: reproduced.** All 12 named changes are positive under every reading: Mmp2, Mmp3 and
Timp1 at 3 M and at 7 M, and Mmp3, Timp1, Timp2, Timp3, Adam10 and Adam17 in 7 M against 3 M 5XFAD.
Only Timp1 at 7 M and Mmp2 at 3 M (separate model) have adjusted p below 0.05; the six 7 M against
3 M changes have adjusted p 0.42 to 0.99 where one is given, and Mmp3 has none.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| 12 IHC claims of Figure 1 and Figures S2 to S6 | blocked, data_missing (no image data in the NWB files) |
| fig1a-pv-pnns-intense-in-5xfad | blocked, data_missing (no image data; depends on fig1g) |
| fig3b, figs9 (2), figs1 (2), fig3g, fig5-5m-pnns-normal, fig5i (2) | blocked, data_missing (no image data) |
| 15 social memory claims (Figures 2, 3, 5, S8) and 4 object recognition claims (Figures 2I to 2K, S7) | blocked, metadata_insufficient (videos without labels, no scored times) |
| fig4a-7m-social-memory-deficit | blocked, data_missing (no behaviour of the RNA-seq cohort) |
| fig4b-3m-degs | blocked, methods_underspecified (readings disagree) |
| fig4c-7m-degs | partially reproduced |
| fig4d-3m-go-memory-terms-down | blocked, methods_underspecified (readings disagree) |
| fig4e-7m-go-immune-terms-up | reproduced |
| fig4fg-ecm-go-terms | blocked, methods_underspecified (readings disagree) |
| fig4h-pnn-synthesis-genes | reproduced |
| fig4i-3m-remodeling-genes-up | reproduced |
| fig4i-7m-remodeling-genes-up | reproduced |
| fig4i-common-and-7m-vs-3m-up | reproduced |

How the analysis differs from the paper's: DESeq2 1.44 in R is replaced by pydeseq2 0.5, with
independent filtering tuned at R's alpha 0.1; it was not compared with R, which the entry's
environment cannot lock. The paper's DESeq2 model, GO enrichment tool, background, term sizes and
GO release are not stated, so every reading the claims file names is run, with two term-size
limits for GO, and a claim whose readings disagree is blocked as methods_underspecified; the GO
release is 2024-06-17. The RNA-seq data are GEO's counts, not data in the dandiset.

What was found about the dataset:
- All 481 NWB files (185,480 bytes each) hold only file-level metadata. No image, ROI, table or
  measurement is deposited, though the description says the dandiset includes "immunohistochemical
  imaging data quantifying PNN integrity".
- The 175 behaviour videos are MP4 files outside NWB with no subject, genotype, treatment, trial or
  phase labels; several arenas share each video. The scored interaction and exploration times are
  not deposited.
- The object recognition videos are deposited four times (Figures 2I, 2J, 2K and S7), and in each
  set Trial 6 and Trial 7 are the same file: 7 names per set, 6 recordings.
- The description says the dandiset includes bulk RNA-seq data; it holds none, and they are in GEO
  GSE317499. The description also keeps a placeholder sentence ("GSEXXXXX").
- The NWB subject is a group label, not an animal (82 subject_ids for 481 files, up to 31 files
  per subject_id); sex is "U" in 476 files although the paper used male mice for behaviour and
  both sexes for IHC.